# Coil & Cog: Wan 2.2 quality test (ROADMAP 3.5)

Makes the same 2 shots as video #5 (cutaway and studio) with the open model **Wan 2.2 TI2V-5B** (Apache 2.0), so you can compare them with your Veo 3.1 Fast clips.

**Before you click Run all** (right-hand panel, *Session options*):
1. **Accelerator: GPU T4 x2.** Not P100: Kaggle's PyTorch no longer supports it.
2. **Internet: On.** It needs your phone number verified in Kaggle settings.

Then **Run all**. The first run downloads the model (~20 GB, several minutes). The clips appear in the *Output* panel as `wan_clips.zip`. Download it and send the clips/screenshots for the comparison.

In [ ]:
# 1. Install and check the GPU
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # less memory fragmentation
!pip install -q --upgrade diffusers transformers accelerate imageio-ffmpeg ftfy
import os
import time
import zipfile

import torch

assert torch.cuda.is_available(), "No GPU: set Accelerator to 'GPU T4 x2' and run again."
name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
print(f"GPU: {name} (compute {major}.{minor}), torch {torch.__version__}")
assert major >= 7, f"{name} is too old for Kaggle's PyTorch: choose 'GPU T4 x2' in Session options."

In [ ]:
# 2. Settings: the two prompts are video #5's Flow prompts (sound line removed, 5 s instead of 8 s)
PROMPTS = {
    "shot_01_cutaway": "Create a premium 5-second technical cutaway video of the alternator, vertical 9:16, with three smooth camera angles.\nStart close on the alternator as its outer casing splits apart along a clean cut plane, revealing Copper wire windings and magnetic rotor.\nThen the camera orbits the opened alternator while Rotor moves as it does in operation, and the shot finishes on a macro close-up of Rotor in motion. Cut surfaces are clean brushed metal.\nMetal housing splits open to reveal copper coils and a central rotor.\nRealistic materials, polished commercial lighting with a cool cyan rim light from the left and a warm amber rim light from the right, smooth camera movement, sharp product detail, photorealistic.\nNo logos, no text, no brand names, no people, no hands, no faces, no extra or distorted components.",
    "shot_04_studio": "Create a premium 5-second product video of the alternator, vertical 9:16, with three smooth camera angles.\nOpen on a close-up of the alternator rotating slowly on a matte dark pedestal against a deep graphite studio background, with crisp metallic highlights sliding across Aluminum housing and cooling fins.\nThen a low angle orbits around the alternator, and the shot finishes on a three-quarter hero angle with a close detail of Aluminum housing and cooling fins.\nComplete alternator unit rotating smoothly on a dark studio pedestal.\nRealistic materials, polished commercial lighting with a cool cyan rim light from the left and a warm amber rim light from the right, smooth camera movement, sharp product detail, photorealistic.\nNo logos, no text, no brand names, no people, no hands, no faces, no extra or distorted components.",
}
NEGATIVE = (
    "blurry, low quality, distorted parts, melted metal, text, letters, logo, watermark, "
    "people, hands, faces, static frame, flicker"
)
WIDTH, HEIGHT = (
    480,
    832,
)  # vertical 9:16-ish; the model's 480p size (720p does not fit a free T4 well)
FRAMES, FPS = 81, 16  # 81 frames at 16 fps = about 5 seconds
FALLBACK_FRAMES = 49  # about 3 seconds, used automatically if 81 frames run out of GPU memory
STEPS = 30  # fewer = faster but rougher
SEED = 42
OUT_DIR = "/kaggle/working/wan_clips"
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# 3. Load the model (first run downloads it)
from diffusers import AutoencoderKLWan, WanPipeline, WanTransformer3DModel
from diffusers.schedulers.scheduling_unipc_multistep import UniPCMultistepScheduler
from transformers import UMT5EncoderModel

MODEL_ID = "Wan-AI/Wan2.2-TI2V-5B-Diffusers"
t0 = time.time()
# The T4 has no fast bfloat16, and in bfloat16 PyTorch falls back to an attention method that
# needs ~6 GB extra. So: the video model in float16 (fast, memory-saving attention on a T4), the
# text encoder in bfloat16 (float16 can overflow there), the VAE in float32 (official advice).
vae = AutoencoderKLWan.from_pretrained(MODEL_ID, subfolder="vae", torch_dtype=torch.float32)
text_encoder = UMT5EncoderModel.from_pretrained(
    MODEL_ID, subfolder="text_encoder", torch_dtype=torch.bfloat16
)
transformer = WanTransformer3DModel.from_pretrained(
    MODEL_ID, subfolder="transformer", torch_dtype=torch.float16
)
pipe = WanPipeline.from_pretrained(
    MODEL_ID, vae=vae, text_encoder=text_encoder, transformer=transformer, torch_dtype=torch.float16
)
pipe.scheduler = UniPCMultistepScheduler.from_config(
    pipe.scheduler.config, flow_shift=3.0
)  # 3.0 = 480p
pipe.enable_model_cpu_offload()  # moves parts to the GPU only while they run, to fit 16 GB
pipe.vae.enable_tiling()
print(f"Model loaded in {time.time() - t0:.0f} s")

In [ ]:
# 4. Generate the clips (falls back to ~3 s clips if 5 s does not fit in GPU memory)
import gc

import numpy as np
from diffusers.utils import export_to_video


def generate(prompt, frames):
    return pipe(
        prompt=prompt,
        negative_prompt=NEGATIVE,
        width=WIDTH,
        height=HEIGHT,
        num_frames=frames,
        num_inference_steps=STEPS,
        guidance_scale=5.0,
        generator=torch.Generator("cpu").manual_seed(SEED),
    ).frames[0]


timings = {}
frames_used = FRAMES
for name, prompt in PROMPTS.items():
    print(f"Generating {name} ({frames_used} frames) ...")
    t0 = time.time()
    try:
        video = generate(prompt, frames_used)
    except torch.OutOfMemoryError:
        torch.cuda.empty_cache()
        frames_used = FALLBACK_FRAMES
        print(f"  out of GPU memory: retrying with {frames_used} frames (~3 s)")
        t0 = time.time()
        video = generate(prompt, frames_used)
    seconds = time.time() - t0
    brightness = float(np.nanmean(video)) if np.isfinite(video).any() else float("nan")
    path = f"{OUT_DIR}/wan_{name}.mp4"
    export_to_video(video, path, fps=FPS)
    timings[name] = round(seconds)
    print(f"  saved {path}: {seconds / 60:.1f} min, mean brightness {brightness:.2f}")
    del video  # free RAM before the next clip (Kaggle restarts the notebook when RAM runs out)
    gc.collect()
    torch.cuda.empty_cache()
    if not np.isfinite(brightness) or brightness < 0.02:
        print("  WARNING: black or broken frames (float16 overflow). Tell Claude; this is fixable.")
print(
    "Minutes per clip:", {k: round(v / 60, 1) for k, v in timings.items()}, "| frames:", frames_used
)

In [ ]:
# 5. Zip for download (Output panel -> wan_clips.zip)
zip_path = "/kaggle/working/wan_clips.zip"
with zipfile.ZipFile(zip_path, "w") as z:
    for f in sorted(os.listdir(OUT_DIR)):
        z.write(os.path.join(OUT_DIR, f), f)
print("Ready:", zip_path, sorted(os.listdir(OUT_DIR)))
print("GPU used:", torch.cuda.get_device_name(0), "| timings (s):", timings)